<a href="https://colab.research.google.com/github/ahmad1bakundi-ops/data-engineering-journey/blob/main/lesson_05_joins.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
DATA_PATH = '/content/drive/MyDrive/data-engineering-journey/lesson-02-data/'
sales = pd.read_csv(DATA_PATH + 'nigerian_ecommerce_sales_CLEAN.csv')
sales['order_date'] = pd.to_datetime(sales['order_date'])
sales['revenue'] = sales['quantity'] * sales['unit_price_ngn']
sales.shape

(724, 12)

In [3]:
customers = pd.read_csv(DATA_PATH + 'nigerian_ecommerce_customers.csv')
customers.shape

(308, 8)

In [4]:
customers.head()

,customer_id,customer_name,email,phone,signup_date,tier,age,gender
0,1135,Gbenga Okonkwo,gbenga.okonkwo@example.com,9152437967,2022-08-30,Silver,58,Female
1,1193,Zainab Chukwu,zainab.chukwu@example.com,9026433008,2023-02-05,Bronze,25,Female
2,1214,Folake Adeleke,folake.adeleke@example.com,9062576667,2024-02-06,Bronze,41,Female
3,1249,Chioma Ibrahim,CHIOMA.IBRAHIM@EXAMPLE.COM,7050192331,2022-02-20,Bronze,48,Male
4,1310,Ahmad Okafor,ahmad.okafor@example.com,8046043515,2022-04-27,Silver,59,Male


In [6]:
merged = sales.merge(customers, on='customer_id', how='left')
merged.shape

(728, 19)

In [8]:
customers[customers.duplicated(subset='customer_id', keep=False)].sort_values('customer_id')

,customer_id,customer_name,email,phone,signup_date,tier,age,gender
108,1224,Bola Aliyu,bola.aliyu@example.com,8137689774,2022-07-14,Bronze,48,Male
159,1224,Bola Aliyu,bola.aliyu@example.com,8137689774,2022-07-14,Bronze,48,Male
73,1264,Damilola Obi,damilola.obi@example.com,8144073173,2022-09-01,Gold,20,Male
224,1264,Damilola Obi,damilola.obi@example.com,8144073173,2022-09-01,Gold,20,Male


In [9]:
customers = customers.drop_duplicates(subset='customer_id')
customers.shape

(306, 8)

In [10]:
merged = sales.merge(customers, on='customer_id', how='left')
merged.shape

(724, 19)

In [11]:
unmatched = merged[merged['tier'].isnull()]
unmatched.shape

(37, 19)

In [12]:
merged = sales.merge(customers, on='customer_id', how='left', indicator=True)
merged['_merge'].value_counts()

,count
_merge,
both,694
left_only,30
right_only,0


In [13]:
merged.groupby('tier').agg(
    order_count=('order_id', 'count'),
    total_revenue=('revenue', 'sum'),
    avg_order_value=('revenue', 'mean'),
).sort_values('avg_order_value', ascending=False)

,order_count,total_revenue,avg_order_value
tier,,,
Bronze,400,42009279.22,105023.198050
Silver,173,16233397.92,93834.670058
Gold,81,6195006.21,76481.558148
Platinum,33,1624892.97,49239.180909


In [14]:
merged_delivered = merged[merged['status'] == 'Delivered']
merged_delivered.groupby('tier')['revenue'].mean().sort_values(ascending=False)

,revenue
tier,
Gold,123404.000714
Silver,117894.414333
Bronze,61869.543933
Platinum,17008.040000
